# RNN(LSTM)实现情感分类


## 1. 数据准备

本项目使用情感分类的经典数据集 [IMDB 影评数据集](https://ai.stanford.edu/~amaas/data/sentiment/)，数据集包含 Positive 和 Negative 两类，下面为其样例：

| Review  | Label  |
|:---|:---:|
| "Quitting" may be as much about exiting a pre-ordained identity as about drug withdrawal. As a rural guy coming to Beijing, class and success must have struck this young artist face on as an appeal to separate from his roots and far surpass his peasant parents' acting success. Troubles arise, however, when the new man is too new, when it demands too big a departure from family, history, nature, and personal identity. The ensuing splits, and confusion between the imaginary and the real and the dissonance between the ordinary and the heroic are the stuff of a gut check on the one hand or a complete escape from self on the other.  |  Negative |
|Story of a man who has unnatural feelings for a pig. Starts out with a opening scene that is a terrific example of absurd comedy. A formal orchestra audience is turned into an insane, violent mob by the crazy chantings of it's singers. Unfortunately it stays absurd the WHOLE time with no general narrative eventually making it just too off putting. Even those from the era should be turned off. The cryptic dialogue would make Shakespeare seem easy to a third grader. On a technical level it's better than you might think with some good cinematography by future great Vilmos Zsigmond. Future stars Sally Kirkland and Frederic Forrest can be seen briefly.| Negative  |
|I saw the capsule comment said "great acting." In my opinion, these are two great actors giving horrible performances, and with zero chemistry with one another, for a great director in his all-time worst effort. Robert De Niro has to be the most ingenious and insightful illiterate of all time. Jane Fonda's performance uncomfortably drifts all over the map as she clearly has no handle on this character, mostly because the character is so poorly written. Molasses-like would be too swift an adjective for this film's excruciating pacing. Although the film's intent is to be an uplifting story of curing illiteracy, watching it is a true "bummer." I give it 1 out of 10, truly one of the worst 20 movies for its budget level that I have ever seen.| Negative  |
| This movie is amazing because the fact that the real people portray themselves and their real life experience and do such a good job it's like they're almost living the past over again. Jia Hongsheng plays himself an actor who quit everything except music and drugs struggling with depression and searching for the meaning of life while being angry at everyone especially the people who care for him most.  | Positive  |
|Bromwell High is a cartoon comedy. It ran at the same time as some other programs about school life, such as "Teachers". My 35 years in the teaching profession lead me to believe that Bromwell High's satire is much closer to reality than is "Teachers". The scramble to survive financially, the insightful students who can see right through their pathetic teachers' pomp, the pettiness of the whole situation, all remind me of the schools I knew and their students. When I saw the episode in which a student repeatedly tried to burn down the school, I immediately recalled ......... at .......... High. A classic line: INSPECTOR: I'm here to sack one of your teachers. STUDENT: Welcome to Bromwell High. I expect that many adults of my age think that Bromwell High is far fetched. What a pity that it isn't!| Positive  |
|If you like adult comedy cartoons, like South Park, then this is nearly a similar format about the small adventures of three teenage girls at Bromwell High. Keisha, Natella and Latrina have given exploding sweets and behaved like bitches, I think Keisha is a good leader. There are also small stories going on with the teachers of the school. There's the idiotic principal, Mr. Bip, the nervous Maths teacher and many others. The cast is also fantastic, Lenny Henry's Gina Yashere, EastEnders Chrissie Watts, Tracy-Ann Oberman, Smack The Pony's Doon Mackichan, Dead Ringers' Mark Perry and Blunder's Nina Conti. I didn't know this came from Canada, but it is very good. Very good!| Positive  |

此外，需要使用预训练词向量对自然语言单词进行编码，以获取文本的语义特征，选取 [Glove](https://nlp.stanford.edu/projects/glove/) 词向量作为 Embedding。

### 1.1 数据下载模块



以下仅展示下载代码，数据集已下载至 `/home/jovyan/work/datasets/68919f50abbdb34700427ae2-momodel` 目录中 

In [ ]:
import os
import shutil
import requests
import tempfile
from tqdm import tqdm
from typing import IO
from pathlib import Path

import random
import numpy as np
import mindspore as ms

def setseed(seed):
    random.seed(seed)
    np.random.seed(seed)
    ms.set_seed(seed)

setseed(1234)

# 指定保存路径
cache_dir = './others'

def http_get(url: str, temp_file: IO):
    """使用requests库下载数据，并使用tqdm库进行流程可视化"""
    req = requests.get(url, stream=True)
    content_length = req.headers.get('Content-Length')
    total = int(content_length) if content_length is not None else None
    progress = tqdm(unit='B', total=total)
    for chunk in req.iter_content(chunk_size=1024):
        if chunk:
            progress.update(len(chunk))
            temp_file.write(chunk)
    progress.close()

def download(file_name: str, url: str):
    """下载数据并存为指定名称"""
    if not os.path.exists(cache_dir):
        os.makedirs(cache_dir)
    cache_path = os.path.join(cache_dir, file_name)
    cache_exist = os.path.exists(cache_path)
    if not cache_exist:
        with tempfile.NamedTemporaryFile() as temp_file:
            http_get(url, temp_file)
            temp_file.flush()
            temp_file.seek(0)
            with open(cache_path, 'wb') as cache_file:
                shutil.copyfileobj(temp_file, cache_file)
    return cache_path


完成数据下载模块后，下载 IMDB 数据集及 glove 预训练词向量

```
imdb_path = download('aclImdb_v1.tar.gz', 'https://mindspore-website.obs.myhuaweicloud.com/notebook/datasets/aclImdb_v1.tar.gz')

glove_path = download('glove.6B.zip', 'https://mindspore-website.obs.myhuaweicloud.com/notebook/datasets/glove.6B.zip')
```


### 1.2 加载IMDB数据集

下载好的 IMDB 数据集为```tar.gz```文件，我们使用 Python 的`tarfile`库对其进行读取，并将所有数据和标签分别进行存放。  
原始的 IMDB 数据集解压目录如下：

```text
    ├── aclImdb
    │   ├── imdbEr.txt
    │   ├── imdb.vocab
    │   ├── README
    │   ├── test
    │   └── train
    │         ├── neg
    │         ├── pos
    ...
```

数据集已分割为 train 和 test 两部分，且每部分包含 neg 和 pos 两个分类的文件夹，因此需分别 train 和 test 进行读取并处理数据和标签。

In [ ]:
import string
translation_table = str.maketrans("", "", string.punctuation)

class IMDBData():
    """IMDB数据集加载器

    加载IMDB数据集并处理为一个Python迭代对象。

    """
    label_map = {
        "pos": 1,
        "neg": 0
    }
    def __init__(self, path, mode="train"):
        self.mode = mode
        self.path = path
        self.docs, self.labels = [], []

        self._load("pos")
        self._load("neg")

    def _load(self, label):
        current_path = self.path + "aclImdb/{}/{}/".format(self.mode, label)
        for file_name in os.listdir(current_path):
            with open(current_path + file_name, 'r', encoding='utf-8') as f:
                self.docs.append(str(f.read().rstrip("\n\r").translate(translation_table).lower()).split())
                self.labels.append([self.label_map[label]])

    def __getitem__(self, idx):
        return self.docs[idx], self.labels[idx]

    def __len__(self):
        return len(self.docs)


完成 IMDB 数据加载器后，加载训练数据集进行测试，输出数据集数量：

In [ ]:
imdb_path = '/home/jovyan/work/datasets/68919f50abbdb34700427ae2-momodel/'
imdb_train = IMDBData(imdb_path, 'train')
len(imdb_train)


将 IMDB 数据集加载至内存并构造为迭代对象后，可以使用 `mindspore.dataset` 提供的 `Generatordataset` 接口加载数据集迭代对象，并进行下一步的数据处理.  
下面封装一个函数将 train 和 test 分别使用 `Generatordataset` 进行加载，并指定数据集中文本和标签的 `column_name` 分别为 `text` 和 `label`:

In [ ]:
import mindspore.dataset as ds

def load_imdb(imdb_path):
    imdb_train = ds.GeneratorDataset(IMDBData(imdb_path, "train"), column_names=["text", "label"], shuffle=True)
    imdb_test = ds.GeneratorDataset(IMDBData(imdb_path, "test"), column_names=["text", "label"], shuffle=False)
    return imdb_train, imdb_test


In [ ]:
imdb_train, imdb_test = load_imdb(imdb_path)


### 1.3 加载预训练词向量


In [ ]:
def load_glove():
    glove_100d_path = os.path.join('/home/jovyan/work/datasets/68919f50abbdb34700427ae2-momodel/glove/', 'glove.6B.100d.txt')

    embeddings = []
    tokens = []
    with open(glove_100d_path, encoding='utf-8', mode='r') as gf:
        for glove in gf:
            word, embedding = glove.split(maxsplit=1)
            tokens.append(word)
            embeddings.append(np.fromstring(embedding, dtype=np.float32, sep=' '))
    # 添加 <unk>, <pad> 两个特殊占位符对应的embedding
    embeddings.append(np.random.rand(100))
    embeddings.append(np.zeros((100,), np.float32))

    vocab = ds.text.Vocab.from_list(tokens, special_tokens=["<unk>", "<pad>"], special_first=False)
    embeddings = np.array(embeddings).astype(np.float32)
    return vocab, embeddings


In [ ]:
vocab, embeddings = load_glove()
len(vocab.vocab())


### 1.4 negetive 词云
<div class='insertContainerBox column'>
<div class='insertItem' align=center><img src="https://imgbed.momodel.cn/neg.png" /></div>
</div>


### 1.5 positive 词云
<div class='insertContainerBox column'>
<div class='insertItem' align=center><img src="https://imgbed.momodel.cn/pos.png" /></div>
</div>


## 2. 数据集预处理



通过加载器加载的 IMDB 数据集进行了分词处理，但不满足构造训练数据的需要，因此要对其进行额外的预处理。其中包含的预处理如下：

- 通过 Vocab 将所有的 Token 处理为 index id。
- 将文本序列统一长度，不足的使用 `<pad>` 补齐，超出的进行截断。



In [ ]:
lookup_op = ds.text.Lookup(vocab, unknown_token='<unk>')
pad_op = ds.transforms.PadEnd([500], pad_value=vocab.tokens_to_ids('<pad>'))
type_cast_op = ds.transforms.TypeCast(ms.float32)


In [ ]:
imdb_train = imdb_train.map(operations=[lookup_op, pad_op], input_columns=['text'])
imdb_train = imdb_train.map(operations=[type_cast_op], input_columns=['label'])

imdb_test = imdb_test.map(operations=[lookup_op, pad_op], input_columns=['text'])
imdb_test = imdb_test.map(operations=[type_cast_op], input_columns=['label'])


In [ ]:
imdb_train, imdb_valid = imdb_train.split([0.7, 0.3])


In [ ]:
imdb_train = imdb_train.batch(64, drop_remainder=True)
imdb_valid = imdb_valid.batch(64, drop_remainder=True)


## 3. 模型构建


首先需要将输入文本(即序列化后的 index id 列表)通过查表转为向量化表示，此时需要使用 `nn.Embedding` 层加载 Glove 词向量；  
然后使用 RNN 循环神经网络做特征提取；最后将 RNN 连接至一个全连接层，即 `nn.Dense`，将特征转化为与分类数量相同的 size，用于后续进行模型优化训练。  
整体模型结构如下：

```text
nn.Embedding -> nn.RNN -> nn.Dense
```

In [ ]:
import math
import mindspore as ms
import mindspore.nn as nn
import mindspore.ops as ops
from mindspore.common.initializer import Uniform, HeUniform

class RNN(nn.Cell):
    def __init__(self, embeddings, hidden_dim, output_dim, n_layers,
                 bidirectional, pad_idx):
        super().__init__()
        vocab_size, embedding_dim = embeddings.shape
        self.embedding = nn.Embedding(vocab_size, embedding_dim, embedding_table=ms.Tensor(embeddings), padding_idx=pad_idx)
        self.rnn = nn.LSTM(embedding_dim,
                           hidden_dim,
                           num_layers=n_layers,
                           bidirectional=bidirectional,
                           batch_first=True)
        weight_init = HeUniform(math.sqrt(5))
        bias_init = Uniform(1 / math.sqrt(hidden_dim * 2))
        self.fc = nn.Dense(hidden_dim * 2, output_dim, weight_init=weight_init, bias_init=bias_init)

    def construct(self, inputs):
        embedded = self.embedding(inputs)
        _, (hidden, _) = self.rnn(embedded)
        hidden = ops.concat((hidden[-2, :, :], hidden[-1, :, :]), axis=1)
        output = self.fc(hidden)
        return output


### 3.1 损失函数与优化器


完成模型主体构建后，首先根据指定的参数实例化网络；然后选择损失函数和优化器。  
针对本节情感分类问题的特性，即预测 Positive 或 Negative 的二分类问题，我们选择 `nn.BCEWithLogitsLoss`(二分类交叉熵损失函数)。

In [ ]:
hidden_size = 256
output_size = 1
num_layers = 2
bidirectional = True
lr = 0.001
pad_idx = vocab.tokens_to_ids('<pad>')

model = RNN(embeddings, hidden_size, output_size, num_layers, bidirectional, pad_idx)
loss_fn = nn.BCEWithLogitsLoss(reduction='mean')
optimizer = nn.Adam(model.trainable_params(), learning_rate=lr)


### 3.2 训练逻辑

在完成模型构建，进行训练逻辑的设计。一般训练逻辑分为一下步骤：

1. 读取一个 Batch 的数据；
2. 送入网络，进行正向计算和反向传播，更新权重；
3. 返回 loss。

In [ ]:
def forward_fn(data, label):
    logits = model(data)
    loss = loss_fn(logits, label)
    return loss

grad_fn = ops.value_and_grad(forward_fn, None, optimizer.parameters)

def train_step(data, label):
    loss, grads = grad_fn(data, label)
    optimizer(grads)
    return loss

def train_one_epoch(model, train_dataset, epoch=0):
    model.set_train()
    total = train_dataset.get_dataset_size()
    loss_total = 0
    step_total = 0
    with tqdm(total=total) as t:
        t.set_description('Epoch %i' % epoch)
        for i in train_dataset.create_tuple_iterator():
            loss = train_step(*i)
            loss_total += loss.asnumpy()
            step_total += 1
            t.set_postfix(loss=loss_total/step_total)
            t.update(1)


### 3.3 评估指标和逻辑

训练逻辑完成后，需要对模型进行评估。即使用模型的预测结果和测试集的正确标签进行对比，求出预测的准确率。  
由于 IMDB 的情感分类为二分类问题，对预测值直接进行四舍五入即可获得分类标签(0 或 1)，然后判断是否与正确标签相等即可。  
下面为二分类准确率计算函数实现：

In [ ]:
def binary_accuracy(preds, y):
    """
    计算每个batch的准确率
    """

    # 对预测值进行四舍五入
    rounded_preds = np.around(ops.sigmoid(preds).asnumpy())
    correct = (rounded_preds == y).astype(np.float32)
    acc = correct.sum() / len(correct)
    return acc


In [ ]:
def evaluate(model, test_dataset, criterion, epoch=0):
    total = test_dataset.get_dataset_size()
    epoch_loss = 0
    epoch_acc = 0
    step_total = 0
    model.set_train(False)

    with tqdm(total=total) as t:
        t.set_description('Epoch %i' % epoch)
        for i in test_dataset.create_tuple_iterator():
            predictions = model(i[0])
            loss = criterion(predictions, i[1])
            epoch_loss += loss.asnumpy()

            acc = binary_accuracy(predictions, i[1])
            epoch_acc += acc

            step_total += 1
            t.set_postfix(loss=epoch_loss/step_total, acc=epoch_acc/step_total)
            t.update(1)

    return epoch_loss / total


### 3.4 模型训练与保存


以下仅给出训练部分代码，具体执行请看 `train.py`



```
num_epochs = 100
best_valid_loss = float('inf')
ckpt_file_name = os.path.join('/home/jovyan/work/results/', 'sentiment-analysis.ckpt')

for epoch in range(num_epochs):
    train_one_epoch(model, imdb_train, epoch)
    valid_loss = evaluate(model, imdb_valid, loss_fn, epoch)

    if valid_loss < best_valid_loss:
        best_valid_loss = valid_loss
        ms.save_checkpoint(model, ckpt_file_name)
```



### 3.5 模型加载与测试


In [ ]:
ckpt_file_name = os.path.join('/home/jovyan/work/results/', 'sentiment-analysis.ckpt')
param_dict = ms.load_checkpoint(ckpt_file_name)
ms.load_param_into_net(model, param_dict)


对测试集打 batch，然后使用 evaluate 方法进行评估，得到模型在测试集上的效果。

In [ ]:
imdb_test = imdb_test.batch(64)
evaluate(model, imdb_test, loss_fn)


## 4. 自定义输入测试


最后我们设计一个预测函数，实现开头描述的效果，输入一句评价，获得评价的情感分类。具体包含以下步骤:

1. 将输入句子进行分词；
2. 使用词表获取对应的 index id 序列；
3. index id 序列转为 Tensor；
4. 送入模型获得预测结果；
5. 打印输出预测结果。

具体实现如下：

In [ ]:
score_map = {
    1: "Positive",
    0: "Negative"
}

def predict_sentiment(model, vocab, sentence):
    model.set_train(False)
    tokenized = sentence.lower().split()
    indexed = vocab.tokens_to_ids(tokenized)
    tensor = ms.Tensor(indexed, ms.int32)
    tensor = tensor.expand_dims(0)
    prediction = model(tensor)
    return score_map[int(np.round(ops.sigmoid(prediction).asnumpy()))]


In [ ]:
predict_sentiment(model, vocab, "This film is terrible")


In [ ]:
predict_sentiment(model, vocab, "This film is great")
